# 05 Evaluation
Notebook placeholder for model evaluation and metrics.

In [1]:
from pathlib import Path
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score, f1_score

project_root = Path(r'c:\Users\svmoo\OneDrive\Documents\GLUCOMA\glaucoma_project')
sys.path.insert(0, str(project_root / 'src'))

from dataset import create_dataloaders, CLASS_NAMES
from model import create_resnet50_model

print('Imports loaded successfully.')
print(f'Project root: {project_root}')

Imports loaded successfully.
Project root: c:\Users\svmoo\OneDrive\Documents\GLUCOMA\glaucoma_project


In [3]:
device = torch.device('cpu')
dataset_root = project_root / 'dataset'
model_path = project_root / 'outputs' / 'models' / 'best_model.pth'

train_loader, val_loader, test_loader, class_weights = create_dataloaders(
    dataset_root=str(dataset_root),
    batch_size=8
)

model = create_resnet50_model(dropout_rate=0.5)

if model_path.exists():
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint)
    model = model.to(device)
    model.eval()
    print(f'Model loaded from: {model_path}')
else:
    model = None
    print(f'Checkpoint not found: {model_path}')
    print('Run src/train.py first to create the trained model checkpoint.')

print(f'Test batches: {len(test_loader)}')

LOADING DATASETS

📁 Training set:
  Found 3936 images in 'normal/' (label=0)
  Found 2366 images in 'glaucoma/' (label=1)
  Total: 6302 images loaded from c:\Users\svmoo\OneDrive\Documents\GLUCOMA\glaucoma_project\dataset\train

📁 Validation set:
  Found 843 images in 'normal/' (label=0)
  Found 507 images in 'glaucoma/' (label=1)
  Total: 1350 images loaded from c:\Users\svmoo\OneDrive\Documents\GLUCOMA\glaucoma_project\dataset\val

📁 Test set:
  Found 845 images in 'normal/' (label=0)
  Found 508 images in 'glaucoma/' (label=1)
  Total: 1353 images loaded from c:\Users\svmoo\OneDrive\Documents\GLUCOMA\glaucoma_project\dataset\test


⚖️  Computing class weights (for imbalanced REFUGE dataset):
  Class 'normal': 3936 samples, weight = 0.8006
  Class 'glaucoma': 2366 samples, weight = 1.3318

DATALOADER SUMMARY
  Batch size:          8
  Training batches:    788
  Validation batches:  169
  Test batches:        170
  Total train images:  6302
  Total val images:    1350
  Total test ima

In [ ]:
def evaluate_model(model, loader, device):
    all_probs = []
    all_labels = []

    if model is None:
        print('Evaluation skipped: model checkpoint is not available yet.')
        return None

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images).squeeze(1)
            probs = outputs.detach().cpu().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(labels.numpy().tolist())

    y_true = np.array(all_labels)
    y_prob = np.array(all_probs)
    y_pred = (y_prob >= 0.5).astype(int)

    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'report': classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred),
        'y_true': y_true,
        'y_prob': y_prob,
    }
    return metrics


test_metrics = evaluate_model(model, test_loader, device)
if test_metrics is not None:
    print('Test Accuracy:', round(test_metrics['accuracy'] * 100, 2))
    print('Test Precision:', round(test_metrics['precision'] * 100, 2))
    print('Test Recall:', round(test_metrics['recall'] * 100, 2))
    print('Test F1:', round(test_metrics['f1'] * 100, 2))
    print('Test ROC-AUC:', round(test_metrics['roc_auc'], 4))
    print('\nClassification Report:\n')
    print(test_metrics['report'])

Evaluation skipped: model checkpoint is not available yet.


In [ ]:
import seaborn as sns

if test_metrics is not None:
    cm = test_metrics['confusion_matrix']
    y_true = test_metrics['y_true']
    y_prob = test_metrics['y_prob']
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = test_metrics['roc_auc']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=axes[0]
    )
    axes[0].set_title('Confusion Matrix')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')

    axes[1].plot(fpr, tpr, label=f'ROC AUC = {roc_auc:.4f}', linewidth=2)
    axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
    axes[1].set_title('ROC Curve')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print('Plotting skipped because evaluation metrics are not available yet.')